In [47]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import chain
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

User Input <br>
    ↓ <br>
[Step 1] → story_params (JSON) <br>
    ↓ <br>
[Step 2] → outline + story_bible (JSON) <br>
    ↓ <br>
[Step 3 loop] → scenes[ ] (array of text) <br>
    ↓ <br>
[Step 4] → reviewed_scenes[ ] <br>
    ↓ <br>
[Step 5] → polished_story (text) <br>
    ↓ <br>
[Step 6] → final packaged output <br>

In [ ]:
model = ChatGroq(model='llama-3.3-70b-versatile', api_key=os.getenv('GROQ_API_KEY'))  # type: ignore


In [49]:
step1_prompt = ChatPromptTemplate.from_messages([
    ('system', """You are a story analyst. Extract structured story parameters from the user's raw input.

Return ONLY a valid JSON object with no preamble or markdown. Use this exact schema:
{{
  "genre": string,          // e.g. "sci-fi", "fantasy", "romance", "thriller"
  "tone": string,           // e.g. "dark", "whimsical", "dramatic", "humorous"
  "protagonist": {{
    "name": string,         // invent one if not given
    "description": string   // 1-2 sentences
  }},
  "setting": string,        // time + place
  "central_conflict": string, // 1 sentence
  "themes": string[],       // 2-4 themes
  "target_length": string   // "short" | "medium" | "long"
}}

If information is missing, make creative inferences that fit the user's intent."""),
    ('user', 'User input: {user_input}')
])
step1_chain = step1_prompt | model | JsonOutputParser()

In [50]:
story_params = step1_chain.invoke({'user_input': 'create a mystery story about a psychopath'})

In [51]:
story_params

{'genre': 'thriller',
 'tone': 'dark',
 'protagonist': {'name': 'Ethan Blackwood',
  'description': 'A charismatic and intelligent psychopath with a hidden agenda, driven by a desire for control and manipulation. With a charming exterior, Ethan conceals his true nature, making him a compelling and unsettling character.'},
 'setting': 'Modern-day London, with its dark alleys and bustling streets',
 'central_conflict': 'A series of gruesome murders takes place, and Ethan becomes the prime suspect, but as the investigation unfolds, it becomes clear that nothing is as it seems, and the truth behind the killings is more complex and sinister than initially thought',
 'themes': ['The blurred lines between good and evil',
  'The dangers of manipulation and control',
  'The darker aspects of human nature',
  'The cat-and-mouse game between hunter and prey'],
 'target_length': 'medium'}

# Step 2

In [ ]:
step2_prompt = ChatPromptTemplate([
    ('system', '''You are a story architect. Using the story parameters below, create a story bible and a
three-act outline.

Return ONLY a valid JSON object with no preamble or markdown. Use this exact schema:
{{
  "title": string,
  "logline": string,  // one compelling sentence summarizing the whole story
  "characters": [
    {{
      "name": string,
      "role": string,       // "protagonist" | "antagonist" | "supporting"
      "description": string,
      "arc": string         // how they change by the end
    }}
  ],
  "world_rules": string[],  // 3-5 rules or facts about this story's world
  "acts": [
    {{
      "act": number,        // 1, 2, or 3
      "summary": string,
      "scenes": [
        {{
          "scene_number": number,
          "title": string,
          "goal": string,       // what this scene must accomplish narratively
          "characters_present": string[]
        }}
      ]
    }}
  ]
}}'''),
    ('user', 'Story parameters: {story_params}')
])

In [53]:
step2_chain = step2_prompt | model | JsonOutputParser()

In [54]:
story_bible = step2_chain.invoke({'story_params': story_params})

In [55]:
story_bible

{'title': 'The Shadow in the Night',
 'logline': 'In modern-day London, a charismatic and intelligent psychopath becomes the prime suspect in a series of gruesome murders, but as the investigation unfolds, the truth behind the killings reveals a complex web of manipulation and control that threatens to destroy everything.',
 'characters': [{'name': 'Ethan Blackwood',
   'role': 'protagonist',
   'description': 'A charismatic and intelligent psychopath with a hidden agenda, driven by a desire for control and manipulation.',
   'arc': "Ethan's true nature is slowly revealed, and his grip on reality begins to slip, forcing him to confront the darkness within himself."},
  {'name': 'Detective Emily Taylor',
   'role': 'supporting',
   'description': "A determined and sharp-minded detective tasked with solving the murders, who becomes increasingly obsessed with uncovering the truth behind Ethan's involvement.",
   'arc': "Emily's fixation on the case takes a toll on her personal life, and s

In [27]:
step3_prompt = ChatPromptTemplate([
    ('system', """You are a creative fiction writer. Write a single vivid scene.

Guidelines:
- Third-person past tense
- Show don't tell
- 200-350 words
- Output only the scene text"""),
    ('user', '''Write a single vivid scene for an ongoing story.

STORY BIBLE (keep everything consistent with this):
{story_params}

SCENES WRITTEN SO FAR (maintain continuity):
{previous_scenes}

NOW WRITE THIS SCENE:
- Title: {sence_title}
- Goal: {scene_goal}
- Characters present: {scene_characters}

Output only the scene text. No labels, no commentary.'''),
    
])

In [67]:
@chain
def step3_chain(params: dict):
    story_bible = params['story_bible']
    story_params = params['story_params']
    all_draft_scenes = []
    for act in story_bible['acts']:
        for scene in act['scenes']:
            all_info = dict()
            all_info['scene_title'] = scene['title']
            all_info['scene_goal'] = scene['goal']
            all_info['scene_characters'] = '\n'.join(scene['characters_present'])

            all_draft_scenes.append(all_info)
    previous_scenes = []
    for scene in all_draft_scenes:
        prompt = step3_prompt.invoke({
            'story_params': story_params,
            'previous_scenes': '\n\n'.join(previous_scenes),
            'sence_title': scene['scene_title'],
            'scene_goal': scene['scene_goal'],
            'scene_characters': scene['scene_characters']
        })
        response = model.invoke(prompt)
        previous_scenes.append(response.content)
    return previous_scenes
            

In [46]:
step4_prompt = ChatPromptTemplate([
    ('system', '''You are a senior fiction editor. Your job is to take a set of raw scenes and turn them
into a seamless, polished story. Your tasks:
1. Write smooth transitions between scenes — no abrupt jumps
2. Unify the narrative voice and prose style throughout
3. Adjust pacing — slow down emotional beats, tighten action
4. Ensure the ending feels earned and resonant

Output the complete polished story as flowing prose only.
No scene labels, no headers — just the story from first word to last.'''),
    ('user', '''STORY BIBLE (for reference):
{story_bible}

RAW SCENES (in order):
{all_scenes}''')
])

In [56]:
step4_chain = step4_prompt | model | StrOutputParser()

In [57]:
full_story = step4_chain.invoke({'story_bible': story_bible, 'all_scenes': '\n\n'.join(all_scenes)})

In [63]:
step5_prompt = ChatPromptTemplate([
    ('system', '''You are a publishing assistant. Take the finished story and package it for presentation. Return ONLY a valid JSON object:
{{
  "title": string,
  "subtitle": string,         // optional tagline
  "blurb": string,            // 2-3 sentence back-cover description, no spoilers
  "content_warnings": string[], // e.g. ["mild violence", "themes of loss"] — empty if none
  "formatted_story": string   // the full story with a proper title header and chapter/scene breaks using "---"
}}'''),
    ('user', '''STORY BIBLE:
{story_bible}

FINISHED STORY:
{polished_story}''')
])

In [64]:
step5_chain = step5_prompt | model | JsonOutputParser()

In [65]:
final = step5_chain.invoke({'story_bible': story_bible,
                            'polished_story': full_story})

In [66]:
final

{'title': 'The Shadow in the Night',
 'subtitle': 'A twisted game of cat and mouse',
 'blurb': 'In the dark underbelly of modern-day London, a charismatic and intelligent psychopath becomes the prime suspect in a series of gruesome murders. As Detective Emily Taylor delves deeper into the case, she finds herself in a twisted game of cat and mouse with the killer. But as the investigation unfolds, the truth behind the killings reveals a complex web of manipulation and control that threatens to destroy everything.',
 'content_warnings': ['mild violence',
  'themes of loss',
  'psychological manipulation'],
 'formatted_story': '# The Shadow in the Night\n---\nEthan Blackwood\'s slender fingers danced across the keyboard, the soft glow of the computer screen illuminating his chiseled features. His eyes, an unsettling shade of blue, sparkled with excitement as he crafted the perfect message. The words flowed effortlessly, a twisted serenade to his future prey. He typed the final sentence, a

In [69]:
@chain
def story_generator(user_input: str):
    story_params = step1_chain.invoke({'user_input': user_input})
    story_bible = step2_chain.invoke({'story_params': story_params})
    all_scenes = step3_chain.invoke({'story_params': story_params, 'story_bible': story_bible})
    polished_story = step4_chain.invoke({'story_bible': story_bible, 'all_scenes': '\n\n'.join(all_scenes)})
    metadata = step5_chain.invoke({'story_bible': story_bible, 'polished_story': polished_story})
    return (polished_story, metadata)

In [70]:
polished_story, metadata = story_generator.invoke('Create a mystery story about a hallucinated psychopath in manga style')

In [72]:
len(polished_story)

14021

In [73]:
metadata

{'title': 'Shattered Reflections',
 'subtitle': 'A descent into madness in modern-day Tokyo',
 'blurb': 'In the neon-lit streets of Tokyo, a troubled high school student must navigate the blurred lines between reality and his own paranoid delusions to uncover the truth behind a series of gruesome murders. As the investigation unfolds, he must confront the darkness within himself and face the demons that have haunted him for so long. Will he be able to distinguish reality from fantasy, or will the truth shatter his reflections forever?',
 'content_warnings': ['mild violence',
  'themes of loss',
  'mental health stigma',
  'trauma'],
 'formatted_story': "# Shattered Reflections\n---\nKaito Yamato's eyes fluttered open, his gaze drifting upward to the cracked ceiling of his small, cluttered room. The faint hum of the city outside seeped through the thin walls, a constant reminder that Tokyo never slept. He lay there, frozen, as the shadows on the walls seemed to twist and writhe like liv

In [74]:
print(polished_story)

Kaito Yamato's eyes fluttered open, his gaze drifting upward to the cracked ceiling of his small, cluttered room. The faint hum of the city outside seeped through the thin walls, a constant reminder that Tokyo never slept. He lay there, frozen, as the shadows on the walls seemed to twist and writhe like living things. The air was heavy with the scent of stale cigarettes and yesterday's ramen.

As he sat up, the sheets tangled around his legs, Kaito's mind began to unravel the threads of his dreams. Fragments of memories, or were they hallucinations, swirled in his head like a maelstrom. He couldn't shake the feeling that something was watching him, lurking just beyond the edge of perception. His heart racing, Kaito threw off the covers and planted his feet firmly on the ground, as if anchoring himself to reality.

The room was a mess of scattered papers, empty energy drink cans, and dog-eared textbooks. A calendar on the wall, once a diligent record of his schedule, now hung limp and f